# Exploratory Data Analysis (EDA): Music Context Understanding
### Course: CSE425 / EEE474 / CSE715 Supervised Neural Networks
**Project**: GNN-Based BERT for Understanding Context from Music

This notebook provides an in-depth exploratory analysis of:
1. Multi-modal dataset distributions (genres, multi-label tags, valence/arousal emotion space)
2. Train / Validation / Test 3-way split distributions (guaranteeing no artist leakage)
3. Log-mel spectrograms and 12-pitch chroma feature representations
4. Music structure graph topology (segment similarity + temporal adjacency)
5. Chord transition network analysis

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

# Set plotting aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

## 1. Load Dataset Splits and Metadata

In [ ]:
metadata_path = "../data/processed/dataset_metadata.json"
if os.path.exists(metadata_path):
    with open(metadata_path, "r", encoding="utf-8") as f:
        dataset_meta = json.load(f)
    print(f"Loaded {len(dataset_meta)} track metadata records.")
else:
    print("Metadata not generated yet. Please run src/dataset.py first.")

## 2. Genre & Tag Distribution Across Train / Val / Test Splits

In [ ]:
if 'dataset_meta' in locals():
    splits = [d['split'] for d in dataset_meta]
    genres = [d['genre'] for d in dataset_meta]
    
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    sns.countplot(x=splits, ax=ax[0], palette="crest")
    ax[0].set_title("Dataset 3-Way Partition (Train / Val / Test)", fontweight="bold")
    ax[0].set_xlabel("Partition Split")
    ax[0].set_ylabel("Number of Tracks")
    
    sns.countplot(x=genres, ax=ax[1], palette="viridis", order=sorted(list(set(genres))))
    ax[1].set_title("Genre Distribution Across Tracks", fontweight="bold")
    ax[1].set_xticklabels(ax[1].get_xticklabels(), rotation=30)
    ax[1].set_xlabel("Musical Genre")
    plt.tight_layout()
    plt.show()

## 3. Valence–Arousal Emotion Space (DEAM Continuous Context)

In [ ]:
if 'dataset_meta' in locals():
    valences = [d['valence'] for d in dataset_meta]
    arousals = [d['arousal'] for d in dataset_meta]
    genres = [d['genre'] for d in dataset_meta]
    
    plt.figure(figsize=(9, 7))
    sns.scatterplot(x=valences, y=arousals, hue=genres, s=90, alpha=0.9, palette="tab10")
    plt.axvline(5.0, color='gray', linestyle='--', alpha=0.6)
    plt.axhline(5.0, color='gray', linestyle='--', alpha=0.6)
    plt.text(7.5, 8.2, "High Arousal / Positive\n(Energetic/Happy)", fontsize=10, ha='center', color='darkgreen')
    plt.text(2.5, 8.2, "High Arousal / Negative\n(Tense/Dark)", fontsize=10, ha='center', color='darkred')
    plt.text(2.5, 2.0, "Low Arousal / Negative\n(Melancholic/Sad)", fontsize=10, ha='center', color='navy')
    plt.text(7.5, 2.0, "Low Arousal / Positive\n(Calm/Peaceful)", fontsize=10, ha='center', color='darkcyan')
    plt.title("Russell's Circumplex Emotion Space (Valence vs. Arousal)", fontsize=13, fontweight="bold")
    plt.xlabel("Valence (Negative -> Positive) [1, 9]")
    plt.ylabel("Arousal (Calm -> Energetic) [1, 9]")
    plt.xlim(1.0, 9.0)
    plt.ylim(1.0, 9.0)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

## 4. Visualizing Music Structure Graphs (Section 3: Segment Similarity + Temporal Adjacency)

In [ ]:
sample_graph_path = "../data/processed/graph_track_0000.json"
if os.path.exists(sample_graph_path):
    with open(sample_graph_path, "r", encoding="utf-8") as f:
        g_data = json.load(f)
    
    G = nx.Graph()
    for i in range(g_data["num_nodes"]):
        G.add_node(i, label=f"Seg {i}")
    
    src_nodes = g_data["edge_index"][0]
    dst_nodes = g_data["edge_index"][1]
    weights = g_data["edge_weight"]
    
    for u, v, w in zip(src_nodes, dst_nodes, weights):
        if u != v:
            G.add_edge(u, v, weight=w)
    
    plt.figure(figsize=(8, 6))
    pos = nx.spring_layout(G, seed=42)
    nx.draw_networkx_nodes(G, pos, node_color="#3498db", node_size=600)
    nx.draw_networkx_edges(G, pos, edge_color="#2c3e50", alpha=0.7, width=2.0)
    nx.draw_networkx_labels(G, pos, font_color="white", font_weight="bold")
    plt.title(f"Music Segment Graph Structure (Track 0000) - {g_data['num_nodes']} Segment Nodes", fontsize=13, fontweight="bold")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Preprocessed graph samples not found. Run dataset preparation first.")